# NarrativeCast · ND3 Task 4 — v4: Full POST Semantic Embedding & Domain Evidence

**Mục tiêu:** khảo sát **toàn bộ POST hợp lệ** giai đoạn 2014–2020 từ Parquet sạch, sử dụng pipeline **đã hoàn thành** `src/embeddings/encode_posts.py` và `embedding_store.py`. Không chọn ngách tự động; không fit narrative clustering chính thức. COMMENT giữ cho bước sau.

**Nguyên tắc:** lọc theo `is_usable_text=True`, `interaction_type=POST`; checkpoint dạng shard; fingerprint nguồn, model và code; không ghi đè v2/v3; không tạo số liệu giả; tải embedding lưu sẵn nếu đúng provenance. `MiniBatchKMeans` phía cuối chỉ để *khám phá sơ bộ chủ đề*; chưa phải narrative clustering ND3.4b.

**Đã đối chiếu:** bảng phân công ND3.1–3 (đã tick), `docs/embedding_model_selection.md`, `docs/embedding_storage.md`, `src/embeddings/encode_posts.py`, `src/embeddings/embedding_store.py`. Không mặc định rằng vector full POST đã được tạo từ trước.

## 1 · Mount Drive, imports, cấu hình

In [1]:
%pip -q install pyarrow transformers sentencepiece scikit-learn
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import os,sys,json,hashlib,time,gc,inspect
import numpy as np,pandas as pd,pyarrow.parquet as pq
ROOT=Path('/content/drive/MyDrive/NarrativeCast')
SOURCE=ROOT/'data/processed/reddit_technology_clean.parquet'
OUT=ROOT/'data/interim/task4/full_post_v4'
SHARDS=OUT/'embeddings_shards'
OUT.mkdir(parents=True,exist_ok=True);SHARDS.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(ROOT))
from src.embeddings.encode_posts import PostEncoder
from src.embeddings.embedding_store import EmbeddingStore, EmbeddingMetadata
MODEL='sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE=64; SHARD_SIZE=8000; SCAN_BATCH=40000; MAX_SEQ=256
assert SOURCE.is_file(),f'MISSING SOURCE: {SOURCE}'
assert (ROOT/'src/embeddings/encode_posts.py').is_file()
assert (ROOT/'src/embeddings/embedding_store.py').is_file()
print('DATA:',SOURCE, 'SIZE_GB:',round(SOURCE.stat().st_size/1e9,3))
print('OUTPUT:',OUT)
print('POST filtering: all available months, not keyword restricted')

Mounted at /content/drive
DATA: /content/drive/MyDrive/NarrativeCast/data/processed/reddit_technology_clean.parquet SIZE_GB: 2.545
OUTPUT: /content/drive/MyDrive/NarrativeCast/data/interim/task4/full_post_v4
POST filtering: all available months, not keyword restricted


## 2 · Audit & immutable source fingerprint

Vì file 2+ GB, fingerprint dùng kích thước + mtime_ns + Parquet metadata và schema; nếu thay đổi sẽ dừng tái sử dụng cache. Có thể bật toàn SHA-256 nếu muốn độ bảo đảm chặt hơn. Không dùng mtime riêng lẻ để tin cache.

In [2]:
import torch
pf=pq.ParquetFile(SOURCE)
required={'post_id','content','publish_datetime_utc','interaction_type','is_usable_text'}
assert required.issubset(set(pf.schema.names)),set(pf.schema.names)
def sha256file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for x in iter(lambda:f.read(4*1024*1024),b''):h.update(x)
 return h.hexdigest()
SOURCE_FINGERPRINT={'size':SOURCE.stat().st_size,'mtime_ns':SOURCE.stat().st_mtime_ns,'rows':pf.metadata.num_rows,'schema':str(pf.schema)}
SOURCE_HASH=hashlib.sha256(json.dumps(SOURCE_FINGERPRINT,sort_keys=True).encode()).hexdigest()
CODE_HASH={f:sha256file(ROOT/'src/embeddings'/f) for f in ['encode_posts.py','embedding_store.py']}
CONFIG={'source_fingerprint':SOURCE_HASH,'source_path':str(SOURCE),'model':MODEL,'max_seq_length':MAX_SEQ,'normalize':True,'code_hashes':CODE_HASH,'shard_size':SHARD_SIZE}
RUN_HASH=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:16]
print('Parquet rows:',pf.metadata.num_rows, 'row groups:',pf.num_row_groups)
print('RUN_HASH:',RUN_HASH, 'GPU:',torch.cuda.is_available())
print('Note: verify source bytewise separately if Drive metadata is unreliable')

Parquet rows: 11597712 row groups: 12
RUN_HASH: e7480e1fd1620bc2 GPU: True
Note: verify source bytewise separately if Drive metadata is unreliable


## 3 · Đọc toàn bộ POST theo streaming, xuất registry có kiểm tra

Thực hiện lọc trên **mọi record**, không chọn mẫu. ID duy nhất bắt buộc; không loại trừ các POST không trùng từ khóa. Ghi registry ra CSV bằng append, để tránh RAM spike; nếu registry đã tồn tại cùng RUN_HASH thì tái sử dụng sau kiểm tra.

In [3]:
REG=OUT/'post_registry.csv'; REG_META=OUT/'post_registry_manifest.json'
if REG.is_file() and REG_META.is_file() and json.loads(REG_META.read_text()).get('run_hash')==RUN_HASH:
 print('REUSE REGISTRY:',REG)
else:
 if REG.exists(): REG.unlink()
 total=valid=0; months=set(); unique=set(); seen_dups=0; first=True
 cols=['post_id','content','publish_datetime_utc','interaction_type','is_usable_text']
 for batch in pf.iter_batches(batch_size=SCAN_BATCH,columns=cols):
  df=batch.to_pandas();total+=len(df)
  keep=df['interaction_type'].astype(str).eq('POST') & df['is_usable_text'].eq(True)
  sub=df.loc[keep,['post_id','content','publish_datetime_utc']].copy()
  sub=sub[sub.post_id.notna() & sub.content.notna() & sub.publish_datetime_utc.notna()]
  sub['post_id']=sub['post_id'].astype(str)
  sub['content']=sub['content'].astype(str)
  sub=sub[sub.post_id.str.len().gt(0) & sub.content.str.strip().ne('')]
  sub['publish_datetime_utc']=pd.to_datetime(sub['publish_datetime_utc'],utc=True,errors='coerce')
  sub=sub[sub.publish_datetime_utc.notna()]
  ids=set(sub.post_id)
  assert len(ids)==len(sub),'Duplicate IDs within source batch'
  assert not (ids & unique), 'Duplicate IDs across source batches'
  unique.update(ids);valid+=len(sub)
  months.update(sub.publish_datetime_utc.dt.strftime('%Y-%m').unique())
  sub.to_csv(REG,mode='w' if first else 'a',index=False,header=first);first=False
  if total % 1000000<SCAN_BATCH: print('scanned',f'{total:,}','POST',f'{valid:,}')
 json.dump({'run_hash':RUN_HASH,'source_rows':total,'post_rows':valid,'unique_ids':len(unique),'months':sorted(months)},open(REG_META,'w'),indent=2)
 del unique;gc.collect()
print(json.loads(REG_META.read_text()))
assert json.loads(REG_META.read_text())['source_rows']==pf.metadata.num_rows
assert json.loads(REG_META.read_text())['months']==pd.period_range('2014-01','2020-12',freq='M').astype(str).tolist(), 'Check coverage - expected 84 months'
print('REGISTRY PASS')

scanned 1,000,000 POST 75,797
scanned 2,000,000 POST 140,528
scanned 3,000,000 POST 200,665
scanned 4,000,000 POST 269,836
scanned 5,000,000 POST 342,799
scanned 6,000,000 POST 414,768
scanned 7,000,000 POST 496,264
scanned 8,000,000 POST 568,113
scanned 9,000,000 POST 627,570
scanned 10,000,000 POST 679,994
scanned 11,000,000 POST 728,013
{'run_hash': 'e7480e1fd1620bc2', 'source_rows': 11597712, 'post_rows': 751607, 'unique_ids': 751607, 'months': ['2014-01', '2014-02', '2014-03', '2014-04', '2014-05', '2014-06', '2014-07', '2014-08', '2014-09', '2014-10', '2014-11', '2014-12', '2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10', '2015-11', '2015-12', '2016-01', '2016-02', '2016-03', '2016-04', '2016-05', '2016-06', '2016-07', '2016-08', '2016-09', '2016-10', '2016-11', '2016-12', '2017-01', '2017-02', '2017-03', '2017-04', '2017-05', '2017-06', '2017-07', '2017-08', '2017-09', '2017-10', '2017-11', '2017-12', '2018-01', '2018-0

## 4 · Cache audit: tiếp tục từ shard đã có, không âm thầm dùng embedding sai mô hình

Mỗi shard lưu bằng `EmbeddingStore.save_npz()` của repo, kèm manifest riêng. Chỉ reuse nếu `run_hash`, ID, thứ tự, checksum và metadata đều đúng. V3 là tập POST+COMMENT lấy mẫu, **không coi như kho full POST**. Có thể bổ sung migration v3 riêng sau khi kiểm provenance, không tự động trộn.

In [4]:
MAN_DIR=OUT/'shard_manifests';MAN_DIR.mkdir(exist_ok=True)
def chunks_from_registry():
 for chunk in pd.read_csv(REG,chunksize=SHARD_SIZE,dtype={'post_id':'string'},keep_default_na=False):
  chunk['publish_datetime_utc']=pd.to_datetime(chunk['publish_datetime_utc'],utc=True,errors='raise')
  yield chunk
# Count expected shards without loading all data
N_POST=int(json.loads(REG_META.read_text())['post_rows'])
N_SHARDS=(N_POST+SHARD_SIZE-1)//SHARD_SIZE
print('POST',N_POST,'shards',N_SHARDS,'vectors FP32 estimated GiB:',round(N_POST*384*4/2**30,2))
def validate_cached(shard_i,df):
 path=SHARDS/f'post_{shard_i:04d}.npz';meta=MAN_DIR/f'post_{shard_i:04d}.json'
 if not (path.exists() and meta.exists()):return False
 m=json.loads(meta.read_text())
 if m.get('run_hash')!=RUN_HASH or m.get('n')!=len(df):return False
 if m.get('sha256')!=sha256file(path):return False
 store=EmbeddingStore.load_npz(path,validate=False)
 ok=(store.validate(strict=True).is_valid and store.embedding_dim==384 and
     np.array_equal(store.post_ids.astype(str),df.post_id.to_numpy(dtype=str)) and
     store.metadata.model_name==MODEL)
 del store;return bool(ok)
ready=sum(validate_cached(i,chunk) for i,chunk in enumerate(chunks_from_registry()))
print('EXISTING VALID SHARDS:',ready,'/',N_SHARDS)

POST 751607 shards 94 vectors FP32 estimated GiB: 1.08
EXISTING VALID SHARDS: 0 / 94


## 5 · Full POST encoding (repo PostEncoder + repo EmbeddingStore)

**Đây là cell tính toán tốn GPU.** Chạy ở Colab GPU T4. Nếu mất kết nối, chạy lại các cell từ đầu: shard hợp lệ tự động bỏ qua. Tên model giữ theo task ND3.1. Không dùng `mock` hoặc `tfidf_svd_debug` cho kết quả chính thức.

In [5]:
import torch
assert torch.cuda.is_available(), 'GPU unavailable. Change runtime to GPU T4 before full encoding.'
encoder=None;done=0;reused=0
for i,df in enumerate(chunks_from_registry()):
 if validate_cached(i,df):
  reused+=len(df)
  if i%10==0:print('REUSE shard',i)
  continue
 if encoder is None:
  encoder=PostEncoder(model_name=MODEL,batch_size=BATCH_SIZE,max_seq_length=MAX_SEQ,device='cuda',normalize_embeddings=True)
  assert encoder.embedding_dim==384
 # Abort if checkpoint exists but invalid, rather than silently overwriting
 path=SHARDS/f'post_{i:04d}.npz';meta_path=MAN_DIR/f'post_{i:04d}.json'
 if path.exists() or meta_path.exists():raise RuntimeError(f'INVALID EXISTING SHARD {i}; inspect manually before deleting: {path}')
 texts=df['content'].astype(str).tolist()
 vec=encoder.encode_texts(texts,show_progress=False)
 assert vec.shape==(len(df),384) and np.isfinite(vec).all()
 ids=df.post_id.to_numpy(dtype=str)
 ts=df.publish_datetime_utc.dt.strftime('%Y-%m-%dT%H:%M:%SZ').to_numpy(dtype=str)
 meta=EmbeddingMetadata(model_name=MODEL,embedding_dim=384,num_vectors=len(df),normalize_embeddings=True,max_seq_length=MAX_SEQ,source_dataset=str(SOURCE),extra={'run_hash':RUN_HASH,'source_fingerprint':SOURCE_HASH,'shard_index':i})
 store=EmbeddingStore(vectors=vec,post_ids=ids,timestamps=ts,metadata=meta)
 assert store.validate(strict=True).is_valid
 tmp=path.with_suffix('.partial.npz')
 store.save_npz(tmp)
 os.replace(tmp,path)
 json.dump({'run_hash':RUN_HASH,'n':len(df),'sha256':sha256file(path),'first_id':ids[0],'last_id':ids[-1]},open(meta_path,'w'),indent=2)
 done+=len(df);print('SAVE',i,'rows',len(df),'new',done,'reused',reused)
 del vec,store,texts;gc.collect()
print('ENCODE PASS:',done,'new +',reused,'reused =',done+reused)
assert done+reused==N_POST

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SAVE 0 rows 8000 new 8000 reused 0
SAVE 1 rows 8000 new 16000 reused 0
SAVE 2 rows 8000 new 24000 reused 0
SAVE 3 rows 8000 new 32000 reused 0
SAVE 4 rows 8000 new 40000 reused 0
SAVE 5 rows 8000 new 48000 reused 0
SAVE 6 rows 8000 new 56000 reused 0
SAVE 7 rows 8000 new 64000 reused 0
SAVE 8 rows 8000 new 72000 reused 0
SAVE 9 rows 8000 new 80000 reused 0
SAVE 10 rows 8000 new 88000 reused 0
SAVE 11 rows 8000 new 96000 reused 0
SAVE 12 rows 8000 new 104000 reused 0
SAVE 13 rows 8000 new 112000 reused 0
SAVE 14 rows 8000 new 120000 reused 0
SAVE 15 rows 8000 new 128000 reused 0
SAVE 16 rows 8000 new 136000 reused 0
SAVE 17 rows 8000 new 144000 reused 0
SAVE 18 rows 8000 new 152000 reused 0
SAVE 19 rows 8000 new 160000 reused 0
SAVE 20 rows 8000 new 168000 reused 0
SAVE 21 rows 8000 new 176000 reused 0
SAVE 22 rows 8000 new 184000 reused 0
SAVE 23 rows 8000 new 192000 reused 0
SAVE 24 rows 8000 new 200000 reused 0
SAVE 25 rows 8000 new 208000 reused 0
SAVE 26 rows 8000 new 216000 reused

## 6 · Full verification: matching IDs, timestamps, vector dimensions, monthly coverage

Quét lại **toàn bộ shard** và xác minh checksum trước khi gọi là PASS. Xuất registry+vector reference, không xuất 1 CSV 1GB.

In [6]:
counts=0;monthly={};shard_rows=[]
for i,df in enumerate(chunks_from_registry()):
 assert validate_cached(i,df),f'SHARD {i} FAIL'
 store=EmbeddingStore.load_npz(SHARDS/f'post_{i:04d}.npz',validate=False)
 assert store.validate(strict=True).is_valid
 assert np.isfinite(store.vectors).all()
 assert np.array_equal(store.timestamps.astype(str),df.publish_datetime_utc.dt.strftime('%Y-%m-%dT%H:%M:%SZ').to_numpy(dtype=str))
 counts+=len(store)
 for k,v in df.publish_datetime_utc.dt.strftime('%Y-%m').value_counts().items():monthly[k]=monthly.get(k,0)+int(v)
 shard_rows.append({'shard':i,'file':str(SHARDS/f'post_{i:04d}.npz'),'rows':len(store),'first_id':store.post_ids[0],'last_id':store.post_ids[-1]})
 del store
assert counts==N_POST
pd.DataFrame(shard_rows).to_csv(OUT/'embedding_shard_index.csv',index=False)
pd.DataFrame(sorted(monthly.items()),columns=['month','post_count']).to_csv(OUT/'post_monthly.csv',index=False)
ACCEPT={'status':'PASS','run_hash':RUN_HASH,'model':MODEL,'embedding_dim':384,'full_post_rows':counts,'source_total_rows':pf.metadata.num_rows,'months':len(monthly),'source_fingerprint':SOURCE_HASH,'comment_embedding_created':False,'scope_selected':False,'task':'ND3.4a full POST discovery evidence'}
(OUT/'full_post_acceptance.json').write_text(json.dumps(ACCEPT,indent=2))
print(json.dumps(ACCEPT,indent=2))

{
  "status": "PASS",
  "run_hash": "e7480e1fd1620bc2",
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "full_post_rows": 751607,
  "source_total_rows": 11597712,
  "months": 84,
  "source_fingerprint": "a469984900de94c4655be620b009363af15562d900212f01ea78560b7f72c5db",
  "comment_embedding_created": false,
  "scope_selected": false,
  "task": "ND3.4a full POST discovery evidence"
}


## 7 · Exploratory themes over ALL POST embeddings (not narrative clustering)

Sử dụng MiniBatchKMeans để xây dựng **bản đồ khám phá chủ đề** trên toàn bộ POST; giữ một số cụm thô nhằm giảm chi phí. Chỉ dùng để chọn các cụm cần đọc, **không** gọi đây là narrative, và chưa tự chốt ngách. Phần này đọc shard theo batch để tránh tạo ma trận tương đồng N×N. `fit` trên toàn thời gian chỉ dùng khám phá hậu nghiệm; tuyệt đối không dùng centroid này cho backtest dự báo ở thời điểm quá khứ.

In [7]:
from sklearn.cluster import MiniBatchKMeans
K=120;RANDOM_SEED=42
# Train on a deterministic, bounded subset distributed across EVERY shard; then assign ALL POST.
rng=np.random.default_rng(RANDOM_SEED);train=[]
for i in range(N_SHARDS):
 store=EmbeddingStore.load_npz(SHARDS/f'post_{i:04d}.npz',validate=False)
 ix=rng.choice(len(store),size=min(500,len(store)),replace=False)
 train.append(np.asarray(store.vectors[ix],dtype=np.float32));del store
X=np.vstack(train);assert len(X)>=K
km=MiniBatchKMeans(n_clusters=K,random_state=RANDOM_SEED,batch_size=4096,n_init=3,max_iter=120)
km.fit(X);print('fit rows:',len(X))
ASSIGN=OUT/'exploratory_post_assignments.csv';first=True;cluster_counts=np.zeros(K,dtype=np.int64)
for i,df in enumerate(chunks_from_registry()):
 store=EmbeddingStore.load_npz(SHARDS/f'post_{i:04d}.npz',validate=False)
 labels=km.predict(store.vectors)
 dist=np.linalg.norm(store.vectors-km.cluster_centers_[labels],axis=1)
 cluster_counts+=np.bincount(labels,minlength=K)
 out=df[['post_id','publish_datetime_utc']].copy();out['exploratory_cluster_id']=labels;out['centroid_distance']=dist
 out.to_csv(ASSIGN,mode='w' if first else 'a',header=first,index=False);first=False
 del store,out
np.savez_compressed(OUT/'exploratory_centroids.npz',centroids=km.cluster_centers_.astype('float32'),run_hash=RUN_HASH)
pd.DataFrame({'cluster_id':np.arange(K),'post_count':cluster_counts}).sort_values('post_count',ascending=False).to_csv(OUT/'exploratory_cluster_sizes.csv',index=False)
assert cluster_counts.sum()==N_POST
print('EXPLORATORY CLUSTER ASSIGNMENT PASS',cluster_counts.sum())

fit rows: 47000
EXPLORATORY CLUSTER ASSIGNMENT PASS 751607


## 8 · Representative POST examples for human review, audit and handoff

Không suy diễn nhãn ngách từ cluster số. Trích ví dụ tiêu biểu cho từng cụm, đánh giá độ nhiễu và đề xuất từ chính văn bản. Kết quả chỉ là **review queue**, không phải tên narrative đã nghiệm thu.

In [8]:
# The assignments and registry have identical order by construction, but verify via merge keys.
assign=pd.read_csv(ASSIGN,usecols=['post_id','exploratory_cluster_id','centroid_distance'])
reg=pd.read_csv(REG,usecols=['post_id','content','publish_datetime_utc'])
assert assign.post_id.is_unique and reg.post_id.is_unique
assert len(assign)==len(reg)==N_POST
review=(assign.sort_values(['exploratory_cluster_id','centroid_distance']).groupby('exploratory_cluster_id').head(12)
        .merge(reg,on='post_id',validate='one_to_one'))
review['review_label']='';review['relevance']='';review['notes']=''
review.to_csv(OUT/'full_post_exploratory_review.csv',index=False)
print('Review examples',len(review),'clusters',review.exploratory_cluster_id.nunique())
print('READ:',OUT/'full_post_exploratory_review.csv')
print('Selected domain: PENDING HUMAN REVIEW — no scope automatically selected.')
del assign,reg,review;gc.collect()

Review examples 1440 clusters 120
READ: /content/drive/MyDrive/NarrativeCast/data/interim/task4/full_post_v4/full_post_exploratory_review.csv
Selected domain: PENDING HUMAN REVIEW — no scope automatically selected.


368

## Kết thúc và nghiệm thu

Gửi ZIP thư mục `data/interim/task4/full_post_v4` **trừ các NPZ lớn** (hoặc chỉ gửi `full_post_acceptance.json`, `post_monthly.csv`, `exploratory_cluster_sizes.csv`, `full_post_exploratory_review.csv`, notebook có output, `embedding_shard_index.csv`). Giữ các NPZ trên Drive. Nếu chưa chạy xong cell 5/6, không tuyên bố full embedding PASS.

**Giới hạn:** 120 cụm trên toàn bộ POST giúp rà soát ngách, không chứng minh đã tìm hết mọi ngách; đánh giá định tính, density, độ phủ và các ứng viên vẫn cần thực hiện trước khi chốt Task 4. Không có kết quả COMMENT ở v4.